# Classifier の比較 (MNIST)

`networks.list_classifiers()` が返す全モデルを同じ条件で比較する。

学習するのは読み出し (`LinearReadout`) だけで、特徴抽出側の重みは全モデルで固定 (非訓練)。
読み出しはリッジ回帰の閉形式解で一発で決まるので、勾配法の学習率やエポック数といった
交絡要因なしに「特徴の良さ」だけを比較できる。

計算は `src/classification.py` の関数をそのまま使う。CLI 実験 (`shell/classification.sh`) と
同じ実装なので、ここで得た数字はそのまま本実験と地続きになる。

## 0. セットアップ

In [ ]:
import os
import sys
import time
import warnings

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
sys.path.append(os.path.abspath(".."))
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F

import networks
from src import classification as C

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# グラフの日本語ラベル用 (無い環境では豆腐になるので、その場合はこの行を消す)
plt.rcParams["font.family"] = "Noto Sans CJK JP"

print("torch:", torch.__version__)
print("device:", DEVICE)
print("classifiers:", networks.list_classifiers())

## 1. 設定

**このセルだけ編集すれば、以降は再実行するだけでよい。**

`MODEL_CONFIGS` の値は `networks.build_classifier(<モデル名>, ...)` にそのまま渡る。
モデルごとに引数名が違う (`units` / `filters` / `num_reservoirs`) 点に注意。
各ハイパーパラメータはリストで層ごとに指定することもできる (例: `"units": [256, 512]`)。

In [ ]:
# ============================== ここを編集する ==============================
SEED = 0

DATA_ROOT = "~/torchvision_datasets"  # 無ければ自動でダウンロードされる
N_TRAIN = 60_000                      # 学習に使う枚数 (MNIST の学習データは 60000)
N_TEST = 10_000                       # 評価に使う枚数 (テストデータは 10000)

BATCH_SIZE = 256                      # 特徴量抽出のバッチサイズ
BETA = 1e-3                           # リッジ回帰の正則化係数
N_LAYER = 1                           # 積む層数 (全モデル共通)

MODEL_CONFIGS = {
    "esn": {
        "units": 512,
        "patch_sizes": (4, 4),
        "connectivity": 0.1,
        "leaky": 0.9,
        "spectral_radius": 0.95,
    },
    "bi_esn": {
        "units": 512,
        "patch_sizes": (4, 4),
        "connectivity": 0.1,
        "leaky": 0.9,
        "spectral_radius": 0.95,
    },
    "bi_esn2d": {
        "units": 512,
        "patch_sizes": (4, 4),
        "connectivity": 0.1,
        "leaky": 0.9,
        "spectral_radius": 0.95,
    },
    "conv2d": {
        "filters": 512,
        "kernel_size": 3,
        "activations": "tanh",
    },
    "reservoir_conv2d": {
        "num_reservoirs": 5,
        "units": 12,  # NOTE: リザバー 1 本あたりの値。特徴量次元は 2 * num_reservoirs * units になる
        "kernel_size": 3,
        "connectivity": 0.5,
        "spectral_radius": 0.95,
    },
}
# ===========================================================================

## 2. データ

In [ ]:
x_train, y_train, x_test, y_test, NUM_CLASSES = C.load_dataset("mnist", DATA_ROOT)

x_train, y_train = x_train[:N_TRAIN], y_train[:N_TRAIN]
x_test, y_test = x_test[:N_TEST], y_test[:N_TEST]

# 読み出しをリッジ回帰で解くための one-hot 目標
y_train_oh = F.one_hot(torch.from_numpy(y_train), NUM_CLASSES).float()

INPUT_SHAPE = tuple(x_train.shape[1:])  # (C, H, W)

print(f"train {tuple(x_train.shape)} {x_train.dtype} / test {tuple(x_test.shape)}")
print(f"input_shape={INPUT_SHAPE}  num_classes={NUM_CLASSES}")
print(f"クラス分布 (train): {np.bincount(y_train)}")

## 3. 比較の実行

各モデルについて「構築 → リッジ回帰で読み出しを決定 → テストで評価」を行う。

In [ ]:
def run(name, kwargs, x_tr, y_tr_oh, n_layer=None, seed=None):
    """1 モデルを構築し、読み出しを解いてテストで評価する。"""
    n_layer = N_LAYER if n_layer is None else n_layer
    seed = SEED if seed is None else seed

    C.set_global_determinism(seed)
    t0 = time.time()

    model = (
        networks.build_classifier(
            name, input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES, n_layer=n_layer, seed=seed, **kwargs
        )
        .to(DEVICE)
        .eval()
    )
    model, _ = C.fit_ridge_readout(model, x_tr, y_tr_oh, BETA, BATCH_SIZE, DEVICE)
    metrics = C.evaluate_model(model, x_test, y_test, NUM_CLASSES, BATCH_SIZE, DEVICE)

    result = {"model": name, "feature_dim": int(model.feature_dim), "elapsed": time.time() - t0, **metrics}

    del model
    torch.cuda.empty_cache()

    return result


results = []
for name, kwargs in MODEL_CONFIGS.items():
    r = run(name, kwargs, x_train, y_train_oh)
    results.append(r)
    print(
        f"{r['model']:<17s} dim={r['feature_dim']:5d}  acc={r['acc']:.4f}  "
        f"macro_f1={r['macro_f1']:.4f}  top5={r['top5']:.4f}  {r['elapsed']:5.1f}s",
        flush=True,
    )

## 4. 結果

In [ ]:
header = f"{'model':<18s}{'feature_dim':>12s}{'acc':>9s}{'macro_f1':>10s}{'top5':>8s}{'time [s]':>10s}"
print(header)
print("-" * len(header))
for r in sorted(results, key=lambda r: -r["acc"]):
    print(
        f"{r['model']:<18s}{r['feature_dim']:>12d}{r['acc']:>9.4f}"
        f"{r['macro_f1']:>10.4f}{r['top5']:>8.4f}{r['elapsed']:>10.1f}"
    )

names = [r["model"] for r in results]
pos = np.arange(len(names))

plt.figure(figsize=(8.5, 3.8))
plt.bar(pos - 0.2, [r["acc"] for r in results], 0.4, label="accuracy")
plt.bar(pos + 0.2, [r["macro_f1"] for r in results], 0.4, label="macro F1")
plt.axhline(1 / NUM_CLASSES, color="crimson", ls="--", lw=1, label=f"偶然一致 ({1 / NUM_CLASSES:.1f})")
plt.xticks(pos, names, rotation=15)
plt.ylabel("スコア")
plt.ylim(0, 1)
plt.title(f"MNIST (train={N_TRAIN}, test={N_TEST}, n_layer={N_LAYER})")
plt.legend()
plt.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

## 5. 特徴量次元をそろえた比較

上の表は各モデルの `feature_dim` が違うので、そのまま比べると「モデルの差」と「次元数の差」が混ざる。
読み出しは線形回帰なので、次元が大きいほど有利になるのは当然である。

そこで目標次元を振り、モデルごとにサイズ引数を合わせて比較する。
`reservoir_conv2d` は特徴量次元が `2 * num_reservoirs * units` で決まるため、
目標にぴったり合わないことがある。横軸は**実際に得られた次元**を使う。

In [ ]:
# ============================== ここを編集する ==============================
FEATURE_DIMS = [64, 128, 256, 512]
SWEEP_N_TRAIN = 20_000  # 点数が多いので学習枚数を減らす
# ===========================================================================


def size_kwargs(name, dim):
    """目標の特徴量次元に合わせて、モデルごとのサイズ引数だけを差し替える。"""
    kwargs = dict(MODEL_CONFIGS[name])

    if name in ("esn", "bi_esn", "bi_esn2d"):
        kwargs["units"] = dim
    elif name == "conv2d":
        kwargs["filters"] = dim
    elif name == "reservoir_conv2d":
        kwargs["units"] = max(1, round(dim / (2 * kwargs["num_reservoirs"])))

    return kwargs


sweep = {name: [] for name in MODEL_CONFIGS}
for dim in FEATURE_DIMS:
    for name in MODEL_CONFIGS:
        r = run(name, size_kwargs(name, dim), x_train[:SWEEP_N_TRAIN], y_train_oh[:SWEEP_N_TRAIN])
        sweep[name].append(r)
        print(f"  dim(目標)={dim:4d} {name:<17s} dim(実際)={r['feature_dim']:5d} acc={r['acc']:.4f}", flush=True)

plt.figure(figsize=(6, 4))
for name, rows in sweep.items():
    plt.plot([r["feature_dim"] for r in rows], [r["acc"] for r in rows], "o-", label=name)
plt.axhline(1 / NUM_CLASSES, color="crimson", ls="--", lw=1)
plt.xscale("log", base=2)
plt.xlabel("特徴量の次元")
plt.ylabel("test accuracy")
plt.title(f"次元をそろえた比較 (train={SWEEP_N_TRAIN})")
plt.legend(fontsize=9)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. `conv2d` が偶然一致レベルになる理由

`conv2d` だけスコアが偶然一致 (0.1) 付近から動かない。これは移植や設定の誤りではなく、
**受容野 3x3 の畳み込み 1 層の出力を画像全体で平均している**ことによる。

`features()` は `(B, filters, H, W)` を `mean(dim=(2, 3))` で潰すので、位置情報が完全に失われる。
リザバー系は系列走査で画像全体を 1 つの状態へ畳み込むため同じ空間平均でも情報が残るが、
局所的な畳み込み 1 層ではそうならない。空間平均の前後で比べると差がはっきり出る。

In [ ]:
# 空間平均なしの次元は filters * H * W になるため、この診断だけ小さい構成で行う
PROBE_FILTERS = 32
PROBE_N = 3_000


def ridge_train_accuracy(Z, y_onehot, y_int, beta):
    """学習データ上でのリッジ回帰の精度。

    NOTE: 次元が標本数を超える場合は双対形 (n x n) で解く。
          そのまま Z^T Z を作ると filters * H * W の 2 乗になり、メモリに載らない。
    """
    Z = Z.double()
    n, d = Z.shape

    if d <= n:
        W = torch.linalg.solve(Z.T @ Z + beta * torch.eye(d, dtype=torch.float64), Z.T @ y_onehot)
        pred = Z @ W
    else:
        K = Z @ Z.T
        pred = K @ torch.linalg.solve(K + beta * torch.eye(n, dtype=torch.float64), y_onehot)

    return float((pred.argmax(1).numpy() == y_int).mean())


name = "conv2d"
probe_config = dict(MODEL_CONFIGS[name], filters=PROBE_FILTERS)

C.set_global_determinism(SEED)
probe = (
    networks.build_classifier(
        name, input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES, n_layer=1, seed=SEED, **probe_config
    )
    .to(DEVICE)
    .eval()
)

with torch.no_grad():
    feature_maps = torch.cat([probe.conv2d(b).cpu() for b in C.iter_batches(x_train[:PROBE_N], BATCH_SIZE, DEVICE)])

y_probe = y_train_oh[:PROBE_N].double()
print(f"学習データ上で線形分離できるか (filters={PROBE_FILTERS}, {PROBE_N} 枚):")
for label, Z in (
    ("空間平均あり (現状の features())", feature_maps.mean(dim=(2, 3))),
    ("空間平均なし (位置情報を保持)", feature_maps.flatten(1)),
):
    acc = ridge_train_accuracy(Z, y_probe, y_train[:PROBE_N], BETA)
    print(f"  {label:34s} 次元={Z.shape[1]:6d}  acc={acc:.4f}")

del probe, feature_maps
torch.cuda.empty_cache()

print("\n層を積むと受容野が広がるため改善する:")
for n_layer in (1, 2, 3):
    r = run(name, MODEL_CONFIGS[name], x_train[:SWEEP_N_TRAIN], y_train_oh[:SWEEP_N_TRAIN], n_layer=n_layer)
    print(f"  n_layer={n_layer}: test acc={r['acc']:.4f}")

## 7. まとめ

In [ ]:
best = max(results, key=lambda r: r["acc"])
print(f"設定: train={N_TRAIN}, test={N_TEST}, n_layer={N_LAYER}, beta={BETA}, seed={SEED}\n")

header = f"{'model':<18s}{'feature_dim':>12s}{'acc':>9s}{'chance 比':>11s}"
print(header)
print("-" * len(header))
for r in sorted(results, key=lambda r: -r["acc"]):
    print(f"{r['model']:<18s}{r['feature_dim']:>12d}{r['acc']:>9.4f}{r['acc'] * NUM_CLASSES:>11.1f}x")

print(f"\n最良: {best['model']} (acc={best['acc']:.4f}, feature_dim={best['feature_dim']})")